# 43. 39번 계층형 RRF 유지: 4채널 가중치 최적화

39번의 **Product/Ruling → Baseline RRF 순위 → Direct/Exclusion RRF 결합** 구조를 유지합니다. 39번과 같은 데이터 195건, 제품군 29개, Outer LOPO, Inner StratifiedGroupKFold 5×5, 고정 HS4 5-class Macro-F1을 사용합니다.

1차: Product/Ruling/Direct 0~1 (0.25 간격), Exclusion -1~1 (0.25 간격). 2차: 1차 최적점 주변 0.1 간격. Direct=top3_mean, Exclusion=max로 고정합니다.

주의: 39번의 `Fixed36` (1,1,0.5,0.5)을 **동일 코드 함수로 재평가**하고, 별도 39번 원본 실험 결과와 일치하는지 확인하세요. Exclusion 음수는 법률 해석이 아니라 순위 감점입니다.


## 39번 대비 변경사항

39번에서 Inner CV가 탐색하던 Direct/Exclusion 집계 방식은 고정하고, 대신 Product/Ruling의 1차 결합 가중치와 Direct/Exclusion의 2차 결합 가중치를 탐색합니다. 첫 단계의 Product/Ruling 가중치가 모두 0인 조합은 제외합니다. 첫 단계의 가중치는 **순위에만** 영향을 주므로 양의 공통배율을 곱한 후보는 중복일 수 있습니다.


## 0. 환경 설정

In [ ]:
!pip -q install -U sentence-transformers scipy kiwipiepy scikit-learn
import re, zipfile, requests
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import LeaveOneGroupOut, StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix
WORK=Path('/content/43_hs4_hierarchical_weight_search')
WORK.mkdir(parents=True,exist_ok=True)
BASE='https://raw.githubusercontent.com/lin1118-code/HS-complete-product/feature/explanatory-note-integration/'
print(WORK)

## 1. 실험 29와 동일한 데이터 파이프라인

In [ ]:
src=requests.get(BASE+'29_discriminative_features_experiment.ipynb').json()
for i in [3,5,7]: exec(''.join(src['cells'][i]['source']),globals())
print('평가 표본:',len(eval_df),'| HS6:',len(labels),'| 제품군:',eval_df.product_family.nunique(),'| HS4:',eval_df.hs4.nunique())
assert len(eval_df)==195 and len(labels)==11 and eval_df.product_family.nunique()==29 and eval_df.hs4.nunique()==5
hs4_labels=sorted(eval_df.hs4.astype(str).unique())
print('HS4 labels:',hs4_labels)
display(eval_df.groupby('hs4').agg(n=('hs4','size'),families=('product_family','nunique')))
display(pd.crosstab(eval_df.product_family,eval_df.hs4))

## 2. 해설서 근거 채널 생성

In [ ]:
kb=pd.read_json(BASE+'hs_knowledge_base.jsonl',lines=True).fillna('')
for c in ['knowledge_id','hs_code','rule_type','document_text','source_sheet','source_page_or_line']:
    if c not in kb: kb[c]=''
    kb[c]=kb[c].astype(str).str.strip()
kb['hs_digits']=kb.hs_code.map(lambda x:re.sub(r'\D','',x))
def clean_kb_text(x):
    lines=[]
    for line in str(x).splitlines():
        if re.match(r'^(규칙_ID|HS코드|자료출처|류|적용범위)\s*:',line.strip()): continue
        lines.append(line.strip())
    return ' '.join(x for x in lines if x)
kb['embedding_text']=kb.document_text.map(clean_kb_text)
direct_rows=[]
exclusion_rows=[]
for _,r in kb.iterrows():
    typ=r.rule_type
    own=r.hs_digits[:4]
    if own in hs4_labels and r.embedding_text and '제외' not in typ:
        direct_rows.append({**r.to_dict(),'candidate_hs4':own,'channel':'direct'})
    if '제외' in typ and r.embedding_text:
        refs=sorted(set(re.findall(r'(?:제\s*)?(\d{4})\s*호',r.document_text)))
        for h in refs:
            if h[:4] in hs4_labels: exclusion_rows.append({**r.to_dict(),'candidate_hs4':h[:4],'channel':'exclusion'})
direct_docs=pd.DataFrame(direct_rows).drop_duplicates(['knowledge_id','candidate_hs4']).reset_index(drop=True)
exclusion_docs=pd.DataFrame(exclusion_rows).drop_duplicates(['knowledge_id','candidate_hs4']).reset_index(drop=True)
print('KB:',len(kb),'| Direct:',len(direct_docs),'| Exclusion links:',len(exclusion_docs))
display(pd.concat([direct_docs.groupby('candidate_hs4').size().rename('direct_docs'),exclusion_docs.groupby('candidate_hs4').size().rename('exclusion_links')],axis=1).reindex(hs4_labels).fillna(0).astype(int))
display(exclusion_docs[['knowledge_id','candidate_hs4','rule_type','source_page_or_line','document_text']])

## 3. 임베딩 생성 및 캐시

In [ ]:
CACHE=WORK/'embeddings_43.npz'
if CACHE.exists():
    z=np.load(CACHE,allow_pickle=True)
    query_emb,product_emb,ruling_emb,kb_direct_emb,kb_exclusion_emb=z['query_emb'],z['product_emb'],z['ruling_emb'],z['kb_direct_emb'],z['kb_exclusion_emb']
else:
    sbert=SentenceTransformer('intfloat/multilingual-e5-base')
    def encode_query(x): return sbert.encode(['query: '+str(v) for v in x],batch_size=16,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
    def encode_passage(x): return sbert.encode(['passage: '+str(v) for v in x],batch_size=16,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
    query_emb=encode_query(eval_df.query_text.tolist())
    product_emb=encode_passage(eval_df.query_text.tolist())
    ruling_emb=encode_passage(eval_df.ruling_text.tolist())
    kb_direct_emb=encode_passage(direct_docs.embedding_text.tolist()) if len(direct_docs) else np.empty((0,768),dtype=float)
    kb_exclusion_emb=encode_passage(exclusion_docs.embedding_text.tolist()) if len(exclusion_docs) else np.empty((0,768),dtype=float)
    np.savez_compressed(CACHE,query_emb=query_emb,product_emb=product_emb,ruling_emb=ruling_emb,kb_direct_emb=kb_direct_emb,kb_exclusion_emb=kb_exclusion_emb)
print(query_emb.shape,product_emb.shape,ruling_emb.shape,kb_direct_emb.shape,kb_exclusion_emb.shape)

## 4. 순위 함수와 고정 검색 캐시

In [ ]:
def aggregate_hs4(scores,codes,method='top3_mean'):
    d=pd.DataFrame({'hs4':np.asarray(codes).astype(str),'score':np.asarray(scores,dtype=float)})
    if method=='max': out=d.groupby('hs4').score.max()
    elif method=='mean': out=d.groupby('hs4').score.mean()
    else: out=d.groupby('hs4').score.apply(lambda x:x.nlargest(min(3,len(x))).mean())
    return out.sort_values(ascending=False)
def rrf_two(a,b,k=60,w=1.0):
    pa={str(c):i+1 for i,c in enumerate(a.index)}
    pb={str(c):i+1 for i,c in enumerate(b.index)}
    return pd.Series({c:1/(k+pa.get(c,99999))+w/(k+pb.get(c,99999)) for c in hs4_labels}).sort_values(ascending=False)
def rrf_ablation(base,direct=None,exclusion=None,wd=0.0,we=0.0,k=60):
    ranks=[base]
    weights=[1.0]
    if direct is not None: ranks.append(direct); weights.append(wd)
    if exclusion is not None: ranks.append(exclusion); weights.append(we)
    pos=[{str(c):i+1 for i,c in enumerate(r.index)} for r in ranks]
    return pd.Series({c:sum(w/(k+p.get(c,99999)) for w,p in zip(weights,pos)) for c in hs4_labels}).sort_values(ascending=False)
def rank_kb(scores,docs,method):
    vals={}
    for c in hs4_labels:
        x=np.asarray(scores)[docs.candidate_hs4.astype(str).to_numpy()==c]
        if len(x)==0: vals[c]=-1.0
        elif method=='max': vals[c]=float(np.max(x))
        else: vals[c]=float(np.mean(np.sort(x)[-min(3,len(x)):]))
    return pd.Series(vals).sort_values(ascending=False)
def baseline_rank(train_idx,q):
    codes=eval_df.iloc[train_idx].hs4.astype(str).to_numpy()
    rp=aggregate_hs4(product_emb[train_idx]@query_emb[q],codes,'top3_mean')
    rr=aggregate_hs4(ruling_emb[train_idx]@query_emb[q],codes,'top3_mean')
    return rrf_two(rp,rr,w=1.0)
METHODS=['max','top3_mean']
WEIGHTS=[0.0,0.5,1.0,2.0]
direct_rank_cache={}
exclusion_rank_cache={}
for q in range(len(eval_df)):
    ds=kb_direct_emb@query_emb[q] if len(direct_docs) else np.array([])
    es=kb_exclusion_emb@query_emb[q] if len(exclusion_docs) else np.array([])
    direct_rank_cache[q]={m:rank_kb(ds,direct_docs,m) for m in METHODS}
    exclusion_rank_cache[q]={m:rank_kb(es,exclusion_docs,m) for m in METHODS}
splits=list(LeaveOneGroupOut().split(eval_df,eval_df.hs4,groups=eval_df.product_family))
print('Outer LOPO:',len(splits))
print('Direct docs:',len(direct_docs),'| Exclusion links:',len(exclusion_docs))

## 5. 반복 Stratified Group K-Fold Inner 튜닝

In [ ]:
INNER_SEEDS=[11,23,37,53,71]
INNER_SPLITS=5
LABELS=np.array(hs4_labels,dtype=str)
COARSE=np.round(np.arange(0,1.001,0.25),2)
EXCL=np.round(np.arange(-1,1.001,0.25),2)
GRID=np.array([(p,r,d,e) for p in COARSE for r in COARSE for d in COARSE for e in EXCL if p+r>0],dtype=np.float64)
FIXED=np.array([1,1,0.5,0.5],dtype=np.float64)
BASELINE=np.array([1,1,0,0],dtype=np.float64)
def rank_vector(series):
    order=list(series.index.astype(str))
    return np.array([1/(60+order.index(c)+1) if c in order else 1/(60+99999) for c in LABELS])
def build_features(train_idx,q):
    codes=eval_df.iloc[train_idx].hs4.astype(str).to_numpy()
    ps=aggregate_hs4(product_emb[train_idx]@query_emb[q],codes,'top3_mean')
    rs=aggregate_hs4(ruling_emb[train_idx]@query_emb[q],codes,'top3_mean')
    return np.stack([rank_vector(ps),rank_vector(rs),rank_vector(direct_rank_cache[q]['top3_mean']),rank_vector(exclusion_rank_cache[q]['max'])])
def score_weights(X,W):
    first=np.einsum('wc,nck->nwk',W[:,:2],X[:,:2,:],optimize=True)
    order=np.argsort(-first,axis=2,kind='stable')
    ranks=np.empty_like(order)
    np.put_along_axis(ranks,order,np.arange(1,len(LABELS)+1)[None,None,:],axis=2)
    base=1/(60+ranks)
    return base+np.einsum('wc,nck->nwk',W[:,2:],X[:,2:,:],optimize=True)
def predict_weights(X,W):
    return np.argmax(score_weights(X,W),axis=2)
def evaluate_weights(X,y,W):
    preds=predict_weights(X,W)
    accuracy=(preds==y[:,None]).mean(axis=0)
    f1=np.zeros(len(W))
    for c in range(len(LABELS)):
        tp=((preds==c)&(y[:,None]==c)).sum(axis=0)
        fp=((preds==c)&(y[:,None]!=c)).sum(axis=0)
        fn=((preds!=c)&(y[:,None]==c)).sum(axis=0)
        f1+=np.divide(2*tp,2*tp+fp+fn,out=np.zeros(len(W)),where=(2*tp+fp+fn)>0)/len(LABELS)
    return f1,accuracy
def inner_data(outer_train):
    inner=eval_df.iloc[outer_train].reset_index(drop=True)
    y=inner.hs4.astype(str).to_numpy()
    groups=inner.product_family.to_numpy()
    repeats=[];coverage=[]
    for seed in INNER_SEEDS:
        splitter=StratifiedGroupKFold(n_splits=INNER_SPLITS,shuffle=True,random_state=seed)
        feat=[];truth=[];validated=set();used=0
        for itr,iva in splitter.split(inner,y,groups):
            gtr,gva=outer_train[itr],outer_train[iva]
            if set(eval_df.iloc[gtr].hs4.astype(str))!=set(hs4_labels):continue
            used+=1
            for q in gva:
                feat.append(build_features(gtr,q))
                truth.append(hs4_labels.index(str(eval_df.iloc[q].hs4)))
                validated.add(int(q))
        if not feat:raise ValueError(f'Inner seed {seed}: no valid split')
        repeats.append((np.stack(feat),np.array(truth,dtype=int)))
        coverage.append({'seed':seed,'valid_splits':used,'validation_rows':len(validated),'coverage_rate':len(validated)/len(outer_train)})
    return repeats,coverage
def score_grid(repeats,weights):
    fs=[];acc=[]
    for X,y in repeats:
        f,a=evaluate_weights(X,y,weights)
        fs.append(f);acc.append(a)
    return np.mean(fs,axis=0),np.mean(acc,axis=0)
def pick(repeats,weights):
    f,a=score_grid(repeats,weights)
    idx=int(np.lexsort((-a,-f))[0])
    return weights[idx],float(f[idx]),float(a[idx])
def fine_grid(center):
    axes=[]
    for i,(lo,hi) in enumerate([(0,1),(0,1),(0,1),(-1,1)]):
        axes.append(np.round(np.arange(max(lo,center[i]-0.25),min(hi,center[i]+0.25)+0.001,0.1),2))
    return np.array([(p,r,d,e) for p in axes[0] for r in axes[1] for d in axes[2] for e in axes[3] if p+r>0],dtype=np.float64)
def verify_fixed(tr,q):
    X=build_features(tr,q)[None,:,:]
    actual=rrf_ablation(baseline_rank(tr,q),direct_rank_cache[q]['top3_mean'],exclusion_rank_cache[q]['max'],0.5,0.5)
    expected=str(actual.index[0])
    got=str(LABELS[int(predict_weights(X,FIXED[None,:])[0,0])])
    if expected!=got:raise AssertionError(f'Fixed39 mismatch q={q}: {expected} vs {got}')
print('Coarse candidates:',len(GRID),'Outer folds:',len(splits))


## 6. Outer LOPO 실행 (5개 Fold씩)

In [ ]:
FOLD_START=1
FOLD_END=5
def fold_path(f):return WORK/f'fold_{f:02d}.csv'
def tuning_path(f):return WORK/f'fold_{f:02d}_tuning.csv'
def coverage_path(f):return WORK/f'fold_{f:02d}_coverage.csv'
for fold,(tr,te) in enumerate(splits,1):
    if not FOLD_START<=fold<=FOLD_END:continue
    if all(p.exists() for p in [fold_path(fold),tuning_path(fold),coverage_path(fold)]):
        print('Fold',fold,'완료 - skip');continue
    if set(eval_df.iloc[tr].hs4.astype(str))!=set(hs4_labels):raise ValueError('Outer train missing class')
    repeats,coverage=inner_data(tr)
    coarse,cf,ca=pick(repeats,GRID)
    fine,ff,fa=pick(repeats,fine_grid(coarse))
    best=coarse if (cf,ca)>=(ff,fa) else fine
    rows=[]
    for q in te:
        verify_fixed(tr,q)
        X=build_features(tr,q)[None,:,:]
        for model,w in [('Baseline_39',BASELINE),('Fixed39_Direct_Exclusion',FIXED),('Coarse_Tuned',coarse),('Fine_Tuned',fine),('Selected_Tuned',best)]:
            prediction=str(LABELS[int(predict_weights(X,w[None,:])[0,0])])
            rows.append({'row_index':int(q),'fold':fold,'heldout_family':str(eval_df.iloc[q].product_family),'true_hs4':str(eval_df.iloc[q].hs4),'model':model,'predicted_hs4':prediction,'product_weight':float(w[0]),'ruling_weight':float(w[1]),'direct_weight':float(w[2]),'exclusion_weight':float(w[3])})
    pd.DataFrame(rows).to_csv(fold_path(fold),index=False,encoding='utf-8-sig')
    pd.DataFrame([{'fold':fold,'stage':stage,'product':float(w[0]),'ruling':float(w[1]),'direct':float(w[2]),'exclusion':float(w[3]),'inner_macro_f1':f,'inner_accuracy':a} for stage,w,f,a in [('coarse',coarse,cf,ca),('fine',fine,ff,fa),('selected',best,*( (cf,ca) if np.array_equal(best,coarse) else (ff,fa) ))]]).to_csv(tuning_path(fold),index=False,encoding='utf-8-sig')
    pd.DataFrame(coverage).assign(outer_fold=fold).to_csv(coverage_path(fold),index=False,encoding='utf-8-sig')
    print(f'Fold {fold}/{len(splits)} | coarse={coarse.tolist()} | fine={fine.tolist()} | selected={best.tolist()}')
print('구간 완료')


## 7. 진행률 및 중간 백업

In [ ]:
done=sorted(int(p.stem.split('_')[1]) for p in WORK.glob('fold_[0-9][0-9].csv'))
print('완료:',len(done),'/',len(splits),'| 누락:',sorted(set(range(1,len(splits)+1))-set(done)))
BACKUP='/content/43_fold_backup.zip'
with zipfile.ZipFile(BACKUP,'w',zipfile.ZIP_DEFLATED) as z:
    for p in WORK.glob('fold_*.csv'):z.write(p,p.name)
    if CACHE.exists():z.write(CACHE,CACHE.name)
from google.colab import files
files.download(BACKUP)


## 8. 중간 백업 복구 (새 런타임에서만 실행)

In [ ]:
from google.colab import files
uploaded=files.upload()
for name in uploaded:
    if name.endswith('.zip'):
        with zipfile.ZipFile(name) as z: z.extractall(WORK)
        print('복구 완료:',name)

## 9. 통합 평가 및 최종 결과 ZIP

In [ ]:
files_list=sorted(WORK.glob('fold_[0-9][0-9].csv'))
missing=sorted(set(range(1,len(splits)+1))-set(int(p.stem.split('_')[1]) for p in files_list))
print('누락:',missing)
if missing:raise ValueError('미완료 Fold가 있습니다.')
pred=pd.concat([pd.read_csv(p,dtype={'true_hs4':str,'predicted_hs4':str}) for p in files_list],ignore_index=True)
if pred.duplicated(['row_index','model']).any():raise ValueError('중복 예측 존재')
summary=pred.groupby('model').apply(lambda g:pd.Series({'n':len(g),'macro_f1':f1_score(g.true_hs4,g.predicted_hs4,labels=hs4_labels,average='macro',zero_division=0),'accuracy':accuracy_score(g.true_hs4,g.predicted_hs4)}),include_groups=False).reset_index().sort_values('macro_f1',ascending=False)
tuning=pd.concat([pd.read_csv(p) for p in sorted(WORK.glob('fold_[0-9][0-9]_tuning.csv'))],ignore_index=True)
coverage=pd.concat([pd.read_csv(p) for p in sorted(WORK.glob('fold_[0-9][0-9]_coverage.csv'))],ignore_index=True)
OUT=WORK/'final';OUT.mkdir(exist_ok=True)
for name,df in [('43_predictions',pred),('43_model_comparison',summary),('43_selected_weights',tuning),('43_coverage',coverage)]:
    df.to_csv(OUT/f'{name}.csv',index=False,encoding='utf-8-sig')
display(summary.round(4))
display(tuning[tuning.stage=='selected'].round(3))
with zipfile.ZipFile('/content/43_hs4_hierarchical_weight_search_results.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.glob('*.csv'):z.write(p,p.name)
from google.colab import files
files.download('/content/43_hs4_hierarchical_weight_search_results.zip')


## 결과 확인

`Fixed39_Direct_Exclusion`의 결과를 기존 39번 `Direct_Plus_Exclusion_Fixed36`과 먼저 비교하세요. 이 둘은 동일한 구조와 가중치입니다. 단, 39번의 `Direct_Plus_Exclusion_Tuned`는 fold마다 선택한 가중치와 집계 방법이 달라 고정36과 같지 않습니다.

`Selected_Tuned`의 Outer CV 성능으로 새 가중치 탐색의 일반화 성능을 판단하세요. 각 fold에서 선택한 가중치가 다르므로 최빈값을 그대로 전체 표본의 검증 성능으로 주장하면 안 됩니다.
